# LLM-Assisted Patch Diffing: Finding the Fix in the Diff

When a project ships a security patch, the patch *is* the disclosure. The diff between the
vulnerable and fixed code shows exactly which check was missing, which boundary wasn't
validated, and which function was at fault. Reverse engineers have long exploited this
asymmetry during the **1-day window** — the gap between a patch landing and everyone deploying it.

This cookbook shows how to use Claude as a **force multiplier for defensive patch triage**: given
a before/after pair of a function, Claude classifies the vulnerability, explains the root cause,
and scores severity — turning hours of manual review into seconds. The goal here is **defensive**:
prioritizing patches, writing detections, and understanding risk faster. We work entirely with a
small, synthetic C example and never produce a working exploit.

**What you'll build:**
1. A structured diff of a vulnerable vs. patched function
2. A prompt that turns Claude into a security-patch analyst returning structured JSON
3. A second validation pass to cut false positives
4. Notes on scaling this to real binaries via decompiler output

> This is an educational defensive-security recipe. It identifies *what class* of bug a patch
> fixes; it does not generate exploits.

## Setup

You'll need the `anthropic` SDK and an API key. Set `ANTHROPIC_API_KEY` in your environment
(never hard-code keys in notebooks).

In [ ]:
%pip install -q anthropic

In [ ]:
import os
import json
import difflib
from anthropic import Anthropic

# API key is read from the environment — do not paste keys into the notebook.
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Use a current model. Check the latest aliases at:
# https://docs.claude.com/en/docs/about-claude/models/overview
MODEL = "claude-sonnet-4-6"


## A worked example: a missing bounds check

Below is a small, synthetic C function in two versions. The "before" version reads a
length field from attacker-controlled input and copies that many bytes without validating
the length against the destination buffer — a classic out-of-bounds read/write. The "after"
version adds the missing check.

This pattern (a length or index used before it's validated) is one of the most common
memory-safety bug classes, and it's exactly what a security patch to a parser tends to fix.

In [ ]:
vulnerable = """
int parse_record(const uint8_t *buf, size_t buf_len, record_t *out) {
    uint16_t name_len = read_u16(buf + 2);
    /* copy the name field into the fixed output buffer */
    memcpy(out->name, buf + 4, name_len);
    out->name[name_len] = '\\0';
    return 0;
}
"""

patched = """
int parse_record(const uint8_t *buf, size_t buf_len, record_t *out) {
    uint16_t name_len = read_u16(buf + 2);
    /* validate the declared length against both the input and the destination */
    if (4 + (size_t)name_len > buf_len || name_len >= sizeof(out->name)) {
        return -EINVAL;
    }
    memcpy(out->name, buf + 4, name_len);
    out->name[name_len] = '\\0';
    return 0;
}
"""

## Structuring the diff for the model

Claude reasons better over a clean unified diff than over two blobs of code. We produce a
standard unified diff and pass it along with both full versions for context.

In [ ]:
def unified_diff(before: str, after: str, path: str = "parse_record.c") -> str:
    return "".join(difflib.unified_diff(
        before.splitlines(keepends=True),
        after.splitlines(keepends=True),
        fromfile=f"a/{path}",
        tofile=f"b/{path}",
    ))

diff = unified_diff(vulnerable, patched)
print(diff)

## Asking Claude to analyze the patch

We give Claude a focused system prompt that frames it as a defensive patch analyst, and we
ask for **structured JSON** so the result is machine-usable (e.g., to feed a triage dashboard).
We request: the affected function, the vulnerability class (CWE), the root cause in one or two
sentences, an exploitability assessment, and a severity rating.

In [ ]:
SYSTEM = """You are a defensive security engineer triaging an upstream security patch.
Given a unified diff and the before/after source of a function, identify what vulnerability
the patch fixes. Be precise and conservative: if the diff is not security-relevant, say so.

Respond ONLY with a JSON object using exactly these keys:
- "function": the affected function name
- "is_security_fix": boolean
- "vulnerability_class": short label plus CWE id if applicable (e.g. "Out-of-bounds read (CWE-125)")
- "root_cause": 1-2 sentence explanation of the missing invariant
- "attacker_input": where the untrusted data enters
- "exploitability": one of "low", "medium", "high", with a brief reason (no exploit code)
- "severity": one of "low", "medium", "high", "critical"
- "detection_idea": one sentence on how a defender might detect or mitigate this
Do not include any prose outside the JSON object."""

def analyze_patch(diff: str, before: str, after: str) -> dict:
    user = f"""Unified diff:
```
{diff}
```

Full BEFORE:
```c
{before}
```

Full AFTER:
```c
{after}
```"""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM,
        messages=[{"role": "user", "content": user}],
    )
    text = resp.content[0].text.strip()
    # The model returns a JSON object; strip any accidental code fences.
    if text.startswith("```"):
        text = text.split("```")[1].lstrip("json").strip()
    return json.loads(text)

analysis = analyze_patch(diff, vulnerable, patched)
print(json.dumps(analysis, indent=2))

## A second pass: validating the finding

Single-shot LLM output can be overconfident. A cheap way to cut false positives is a
**verification pass**: hand Claude its own conclusion and the diff, and ask it to argue the
*opposite* — could this be a benign refactor? If the model can't mount a credible case that
it's non-security, our confidence goes up.

In [ ]:
def validate_finding(diff: str, finding: dict) -> dict:
    user = f"""A first-pass analysis claims this patch is a security fix:

{json.dumps(finding, indent=2)}

Here is the diff again:
```
{diff}
```

Play devil's advocate. Could this plausibly be a NON-security change (refactor, style,
performance) rather than a vulnerability fix? Respond ONLY with JSON:
- "could_be_benign": boolean
- "counter_argument": one sentence
- "confidence_fix_is_real": one of "low", "medium", "high"."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": user}],
    )
    text = resp.content[0].text.strip()
    if text.startswith("```"):
        text = text.split("```")[1].lstrip("json").strip()
    return json.loads(text)

validation = validate_finding(diff, analysis)
print(json.dumps(validation, indent=2))

## Scaling to real targets

The example above uses source, but the same pipeline works on **compiled** code, which is the
common 1-day scenario:

1. **Acquire** the pre- and post-patch binaries (vendor update packages, package archives).
2. **Decompile** both with a headless decompiler (Ghidra, IDA) to get C-like pseudocode.
3. **Diff** the changed functions (BinDiff / Diaphora identify which functions changed).
4. **Feed** each changed function pair through `analyze_patch` exactly as above.
5. **Triage** by severity and route the high-confidence, high-severity findings to humans.

A few practical notes:
- Decompiler pseudocode is noisy; passing the unified diff plus both full functions (as we do)
  gives Claude enough context to ignore renamed temporaries and focus on semantic changes.
- Batch many function pairs and sort by the model's `severity` + `confidence` to focus review.
- Keep a human in the loop for anything you act on. This pipeline **prioritizes** review; it
  does not replace the analyst.

## Responsible use

This recipe is for **defenders**: triaging patches, building detections, and understanding risk
during the 1-day window. It deliberately stops at classification and never asks the model to
produce a working exploit. Treat findings as leads to verify, not ground truth.

## Summary

You built a small but complete patch-triage pipeline: a structured diff, a JSON-returning
analyst prompt, and a validation pass — the core of turning a raw security patch into an
actionable, severity-ranked finding in seconds.